In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "CustomerID": [301, 302, 303, 304, 305, 306, 307, 308],
    "Age": [25, 34, np.nan, 29, 150, 41, 38, np.nan],
    "City": [
        "New York", "boston", "New York", "CHICAGO",
        "Boston", None, " chicago ", "Boston"
    ],
    "PurchaseAmount": [120, 250, 180, np.nan, 5000, 320, -50, 410],
    "Satisfaction": [4.5, np.nan, 3.5, 4.8, 4.2, 3.8, np.nan, 4.1]
})

df

,CustomerID,Age,City,PurchaseAmount,Satisfaction
0,301,25.0,New York,120.0,4.5
1,302,34.0,boston,250.0,NaN
2,303,NaN,New York,180.0,3.5
3,304,29.0,CHICAGO,NaN,4.8
4,305,150.0,Boston,5000.0,4.2
5,306,41.0,NaN,320.0,3.8
6,307,38.0,chicago,-50.0,NaN
7,308,NaN,Boston,410.0,4.1


Task 1: Inspect data quality
1. Count missing values in each column.
2. Count duplicate rows.
3. Display rows with at least one missing value.

In [2]:
df.isna().sum()

CustomerID        0
Age               2
City              1
PurchaseAmount    1
Satisfaction      2
dtype: int64

In [3]:
df.duplicated().sum()

np.int64(0)

In [4]:
df[df.isna().any(axis=1)]

,CustomerID,Age,City,PurchaseAmount,Satisfaction
1,302,34.0,boston,250.0,NaN
2,303,NaN,New York,180.0,3.5
3,304,29.0,CHICAGO,NaN,4.8
5,306,41.0,NaN,320.0,3.8
6,307,38.0,chicago,-50.0,NaN
7,308,NaN,Boston,410.0,4.1


Task 2: Clean missing values
4. Replace missing ages with the median of valid ages, excluding the impossible age of 150.
5. Replace missing purchase amounts with the median of valid, non-negative purchases, excluding the amount of 5000.
6. Replace missing satisfaction scores with the column mean.

In [5]:
valid_ages = df.loc[df["Age"].between(18, 100), "Age"]
age_median = valid_ages.median()

df["Age"] = df["Age"].fillna(age_median)

print(age_median)
print(df["Age"])

34.0
0     25.0
1     34.0
2     34.0
3     29.0
4    150.0
5     41.0
6     38.0
7     34.0
Name: Age, dtype: float64


In [6]:
valid_purchases = df.loc[
    df["PurchaseAmount"].between(0, 1000),
    "PurchaseAmount"
]

purchase_median = valid_purchases.median()

df["PurchaseAmount"] = df["PurchaseAmount"].fillna(
    purchase_median
)

print(purchase_median)
print(df["PurchaseAmount"])

250.0
0     120.0
1     250.0
2     180.0
3     250.0
4    5000.0
5     320.0
6     -50.0
7     410.0
Name: PurchaseAmount, dtype: float64


In [7]:
satisfaction_mean = df["Satisfaction"].mean()

df["Satisfaction"] = df["Satisfaction"].fillna(
    satisfaction_mean
)

print(satisfaction_mean)

4.1499999999999995


Task 3: Standardize text
7. Standardize city names using title case and remove extra spaces.
8. Count customers by city after standardization.

In [8]:
df["City"] = df["City"].str.strip().str.title()

print(df["City"])

0    New York
1      Boston
2    New York
3     Chicago
4      Boston
5         NaN
6     Chicago
7      Boston
Name: City, dtype: str


In [9]:
df["City"].value_counts()

City
Boston      3
New York    2
Chicago     2
Name: count, dtype: int64

Task 4: Validate and filter
9. Flag invalid ages (less than 18 or greater than 100).
10. Flag invalid purchases (negative or greater than 1000).
11. Create a cleaned DataFrame excluding rows with invalid ages or purchases.
12. Reset the cleaned DataFrame's index.

In [25]:
df["InvalidAge"] = ~df["Age"].between(18, 100)
#print(df[["CustomerID", "InvalidAge"]])

#df[df["InvalidAge"]][["CustomerID", "InvalidAge"]]
df.loc[df["InvalidAge"] == True, ["CustomerID", "InvalidAge"]]

,CustomerID,InvalidAge
4,305,True


In [26]:
df["InvalidPurchase"] = ~df["PurchaseAmount"].between(0, 1000)
#print(df[["CustomerID", "InvalidPurchase"]])
df.loc[df["InvalidPurchase"] == True, ["CustomerID", "InvalidPurchase"]]


,CustomerID,InvalidPurchase
4,305,True
6,307,True


In [22]:
cleaned_df = df[
    ~df["InvalidAge"] & ~df["InvalidPurchase"]
].copy()

cleaned_df

,CustomerID,Age,City,PurchaseAmount,Satisfaction,InvalidAge,InvalidPurchase
0,301,25.0,New York,120.0,4.50,False,False
1,302,34.0,Boston,250.0,4.15,False,False
2,303,34.0,New York,180.0,3.50,False,False
3,304,29.0,Chicago,250.0,4.80,False,False
5,306,41.0,NaN,320.0,3.80,False,False
7,308,34.0,Boston,410.0,4.10,False,False


In [23]:
cleaned_df = cleaned_df.reset_index(drop=True)

cleaned_df

,CustomerID,Age,City,PurchaseAmount,Satisfaction,InvalidAge,InvalidPurchase
0,301,25.0,New York,120.0,4.50,False,False
1,302,34.0,Boston,250.0,4.15,False,False
2,303,34.0,New York,180.0,3.50,False,False
3,304,29.0,Chicago,250.0,4.80,False,False
4,306,41.0,NaN,320.0,3.80,False,False
5,308,34.0,Boston,410.0,4.10,False,False
